# 🤖 RAG Agentic Chatbot
### LlamaIndex (Vector Store/Retriever) + LangChain Agent + Multi-Tool Support

**Architecture:**
- 📚 **LlamaIndex** → Document ingestion, vector store & fast retrieval
- 🦜 **LangChain** → Agent orchestration & tool management
- 🔧 **Tools:** Web Search, Wikipedia, ArXiv, RAG Retrieval
- 🧠 **LLM:** Groq Llama-3 8B (free tier, swappable)

## 🔑 1. Configuration & API Keys

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # Load from .env file if present

# ── Set your Groq API key here or in a .env file ─────────────────────────────
# Get a FREE key at: https://console.groq.com
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")
# ─────────────────────────────────────────────────────────────────────────────

# Model config
LLM_MODEL = "llama-3.1-8b-instant"
EMBED_MODEL     = "BAAI/bge-small-en-v1.5"  # Free local HuggingFace embedding
CHUNK_SIZE      = 512
CHUNK_OVERLAP   = 64
TOP_K           = 5

print("✅ Configuration loaded")


✅ Configuration loaded


## 📚 2. Document Ingestion with LlamaIndex

In [2]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Document, Settings, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding  # free local embeddings
from llama_index.llms.groq import Groq as LlamaGroq                  # free Groq LLM
from llama_index.vector_stores.faiss import FaissVectorStore
import faiss

# ── Global LlamaIndex settings ────────────────────────────────────────────────
Settings.llm         = LlamaGroq(model=LLM_MODEL, temperature=0)
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL)   # runs locally
Settings.node_parser = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

print("✅ LlamaIndex settings configured (Groq LLM + HuggingFace embeddings)")


c:\Users\Lenovo-T460s\Desktop\Rag_Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LlamaIndex settings configured (Groq LLM + HuggingFace embeddings)


In [3]:
# ── Option A: Load from a directory ──────────────────────────────────────────
# documents = SimpleDirectoryReader("./docs").load_data()

# ── Option B: Use sample in-memory documents (demo) ──────────────────────────
sample_texts = [
    """Retrieval-Augmented Generation (RAG) is a technique that combines the strengths of 
    large language models with external knowledge retrieval. In RAG, relevant documents are 
    fetched from a vector store based on the user query, and then passed as context to the LLM 
    to generate a grounded, accurate response. This reduces hallucinations and keeps information 
    up-to-date without expensive retraining.""",

    """LlamaIndex (formerly GPT Index) is a data framework for building LLM-powered applications. 
    It provides connectors to ingest data from various sources (PDFs, APIs, databases), 
    efficient indexing using vector stores like FAISS or Pinecone, and query engines with 
    sub-question decomposition, re-ranking, and hybrid search. LlamaIndex integrates seamlessly 
    with LangChain for agentic workflows.""",

    """LangChain is a framework for developing applications powered by language models. 
    It enables building agents that can reason step-by-step using the ReAct (Reasoning + Acting) 
    paradigm. Agents decide which tools to call — such as search, calculators, or databases — 
    based on the user's goal, execute actions, observe results, and iterate until a final 
    answer is reached.""",

    """FAISS (Facebook AI Similarity Search) is a library for efficient similarity search 
    and clustering of dense vectors. It supports billion-scale vector search and provides 
    multiple index types: Flat (exact), IVF (approximate), and HNSW (graph-based). 
    FAISS is highly optimized for GPU and CPU and is widely used as the backbone 
    for vector databases in RAG systems.""",

    """Agentic AI refers to systems where an LLM autonomously plans and executes multi-step 
    tasks by calling tools, APIs, or sub-agents. Unlike a single-turn chatbot, an agentic 
    system maintains state across steps, backtracks on failure, and selects the best 
    strategy from multiple options. Key components include a planner, a tool executor, 
    memory, and an evaluator.""",
]

documents = [Document(text=t) for t in sample_texts]
print(f"✅ Loaded {len(documents)} documents")

✅ Loaded 5 documents


## 🗃️ 3. Build FAISS Vector Store with LlamaIndex

In [4]:
import numpy as np

# ── Build FAISS index ─────────────────────────────────────────────────────────
EMBED_DIM = 384   # BAAI/bge-small-en-v1.5 dimension

faiss_index    = faiss.IndexFlatL2(EMBED_DIM)          # Exact L2 search
vector_store   = FaissVectorStore(faiss_index=faiss_index)
storage_ctx    = StorageContext.from_defaults(vector_store=vector_store)

# ── Index documents ───────────────────────────────────────────────────────────
print("⏳ Embedding & indexing documents...")
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_ctx,
    show_progress=True,
)
print(f"✅ Vector store built — {faiss_index.ntotal} vectors indexed")

⏳ Embedding & indexing documents...


Generating embeddings: 100%|██████████| 5/5 [00:02<00:00,  2.37it/s]

✅ Vector store built — 5 vectors indexed


In [5]:
# ── Persist index to disk (optional) ─────────────────────────────────────────
PERSIST_DIR = "./storage"
index.storage_context.persist(persist_dir=PERSIST_DIR)
print(f"✅ Index persisted to {PERSIST_DIR}")

# ── To reload later: ─────────────────────────────────────────────────────────
# from llama_index.core import load_index_from_storage
# storage_ctx2 = StorageContext.from_defaults(
#     vector_store=FaissVectorStore.from_persist_dir(PERSIST_DIR),
#     persist_dir=PERSIST_DIR
# )
# index = load_index_from_storage(storage_ctx2)

✅ Index persisted to ./storage


## 🔧 4. Define LangChain Tools

In [7]:
from langchain_core.tools import tool
from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun

# ════════════════════════════════════════════════════════════════
# TOOL 1 — RAG Retrieval
# ════════════════════════════════════════════════════════════════

retriever = index.as_retriever(similarity_top_k=TOP_K)

@tool
def rag_knowledge_base(query: str) -> str:
    """
    Search the local knowledge base using semantic similarity.
    Use this FIRST for any question about RAG, LlamaIndex,
    LangChain, FAISS, agentic AI, or ingested documents.
    """
    nodes = retriever.retrieve(query)

    if not nodes:
        return "No relevant documents found in the knowledge base."

    chunks = []

    for i, node in enumerate(nodes, start=1):
        score = round(node.score, 4) if node.score else "N/A"

        chunks.append(
            f"[Chunk {i} | Score: {score}]\n"
            f"{node.get_content().strip()}"
        )

    return "\n\n".join(chunks)


# ════════════════════════════════════════════════════════════════
# TOOL 2 — DuckDuckGo Web Search
# ════════════════════════════════════════════════════════════════

ddg = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """
    Search the web for current events, news,
    or information not present in the knowledge base.
    """
    return ddg.run(query)


# ════════════════════════════════════════════════════════════════
# TOOL 3 — Wikipedia
# ════════════════════════════════════════════════════════════════

wiki = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=3,
        doc_content_chars_max=2000,
    )
)

@tool
def wikipedia(query: str) -> str:
    """
    Search Wikipedia for factual and encyclopedic information.
    """
    return wiki.run(query)


# ════════════════════════════════════════════════════════════════
# TOOL 4 — Arxiv
# ════════════════════════════════════════════════════════════════

import arxiv as arxiv_lib
import time

@tool
def arxiv_search(query: str) -> str:
    """Search ArXiv for academic research papers."""
    try:
        client = arxiv_lib.Client(
            page_size=3,
            delay_seconds=5,    # ✅ wait 5s between retries
            num_retries=2,
        )
        search = arxiv_lib.Search(
            query=query,
            max_results=3,
            sort_by=arxiv_lib.SortCriterion.Relevance,
        )
        results = list(client.results(search))

        if not results:
            return "No papers found."

        output = []
        for r in results:
            output.append(
                f"**{r.title}**\n"
                f"Authors: {', '.join(a.name for a in r.authors[:3])}\n"
                f"Published: {r.published.date()}\n"
                f"Summary: {r.summary[:300]}...\n"
                f"URL: {r.entry_id}"
            )
        return "\n\n---\n\n".join(output)

    except Exception as e:
        return f"ArXiv search temporarily unavailable (rate limited). Try again in a minute. Error: {str(e)}"


# Register tools

tools = [
    rag_knowledge_base,
    web_search,
    wikipedia,
    arxiv_search,
]

print(f"✅ {len(tools)} tools registered")

for tool in tools:
    print(tool.name)

✅ 4 tools registered
rag_knowledge_base
web_search
wikipedia
arxiv_search


## 🧠 5. Build the LangChain ReAct Agent

In [8]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
# from langchain_core.prompts import PromptTemplate

from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

# ─────────────────────────────────────────────────────────────
# LLM
# ─────────────────────────────────────────────────────────────
llm = ChatGroq(
    model=LLM_MODEL,
    temperature=0,
    streaming=False,          # ✅ disable streaming (causes issues with tool calls on Groq)
    model_kwargs={
        "parallel_tool_calls": False   # ✅ one tool at a time, more stable
    }
)

# ─────────────────────────────────────────────────────────────
# Memory
# ─────────────────────────────────────────────────────────────
memory = InMemorySaver()

# ─────────────────────────────────────────────────────────────
# System Prompt
# ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a helpful research assistant with access to four tools.

INSTRUCTIONS:
1. Always call rag_knowledge_base FIRST for any question.
2. Read the tool output carefully.
3. Write a complete, detailed answer based on what the tool returned.
4. End your response by mentioning which tool(s) you used.
5. If a tool returns an error or rate limit message, use other available tools or answer from what you already know.

IMPORTANT: Never stop after calling a tool. Always write a full answer using the tool results."""

# prompt = PromptTemplate.from_template(SYSTEM_PROMPT)

# ─────────────────────────────────────────────────────────────
# Create LangGraph ReAct Agent
# ─────────────────────────────────────────────────────────────
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
    checkpointer=memory,
)

print("✅ LangGraph ReAct Agent ready")

✅ LangGraph ReAct Agent ready


C:\Users\Lenovo-T460s\AppData\Local\Temp\ipykernel_5128\3809000174.py:44: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## 💬 6. Chat Helper Function

In [9]:
from IPython.display import Markdown, display
import uuid

def chat(query: str, show_steps: bool = True) -> str:
    """
    Send a query to the agentic chatbot and return the answer.
    
    Args:
        query:      The user's question.
        show_steps: If True, prints intermediate reasoning steps.
    
    Returns:
        Final answer string.
    """
    print(f"\n{'='*60}")
    print(f"🧑 User: {query}")
    print(f"{'='*60}")

    result = agent.invoke(
    {"messages": [HumanMessage(content=query)]},   # ✅ correct key
    config={"configurable": {"thread_id": str(uuid.uuid4())}})
    
    answer = result["messages"][-1].content

    if show_steps and result.get("intermediate_steps"):
        print("\n📋 Reasoning Steps:")
        for i, (action, observation) in enumerate(result["intermediate_steps"], 1):
            print(f"  Step {i}: [{action.tool}] {action.tool_input!r}")
            print(f"  ↳ {str(observation)[:200]}...\n")

    print(f"\n🤖 Agent Answer:")
    display(Markdown(answer))
    return answer

## 🧪 7. Test the Agent

In [10]:
# Test 1 — RAG retrieval from local knowledge base
_ = chat("What is Retrieval-Augmented Generation and how does LlamaIndex help with it?")


🧑 User: What is Retrieval-Augmented Generation and how does LlamaIndex help with it?

🤖 Agent Answer:


Retrieval-Augmented Generation (RAG) is a technique that combines the strengths of large language models with external knowledge retrieval. In RAG, relevant documents are fetched from a vector store based on the user query, and then passed as context to the LLM to generate a grounded, accurate response. This reduces hallucinations and keeps information up-to-date without expensive retraining.

LlamaIndex is a data framework for building LLM-powered applications. It provides connectors to ingest data from various sources (PDFs, APIs, databases), efficient indexing using vector stores like FAISS or Pinecone, and query engines with sub-question decomposition, re-ranking, and hybrid search. LlamaIndex integrates seamlessly with LangChain for agentic workflows.

FAISS is a library for efficient similarity search and clustering of dense vectors. It supports billion-scale vector search and provides multiple index types: Flat (exact), IVF (approximate), and HNSW (graph-based). FAISS is highly optimized for GPU and CPU and is widely used as the backbone for vector databases in RAG systems.

LangChain is a framework for developing applications powered by language models. It enables building agents that can reason step-by-step using the ReAct (Reasoning + Acting) paradigm. Agents decide which tools to call — such as search, calculators, or databases — based on the user's goal, execute actions, observe results, and iterate until a final answer is reached.

Agentic AI refers to systems where an LLM autonomously plans and executes multi-step tasks by calling tools, APIs, or sub-agents. Unlike a single-turn chatbot, an agentic system maintains state across steps, backtracks on failure, and selects the best strategy from multiple options. Key components include a planner, a tool executor, memory, and an evaluator.

I used rag_knowledge_base to answer this question.

In [11]:
# Test 2 — Wikipedia for background knowledge
_ = chat("Give me a brief history of transformer models in NLP.")


🧑 User: Give me a brief history of transformer models in NLP.

🤖 Agent Answer:


I used rag_knowledge_base and web_search.

In [12]:
# Test 3 — ArXiv for recent research papers
_ = chat("What are the latest research papers on RAG (Retrieval Augmented Generation)?")


🧑 User: What are the latest research papers on RAG (Retrieval Augmented Generation)?

🤖 Agent Answer:


I used the following tools:

1. rag_knowledge_base
2. arxiv_search

In [13]:
# Test 4 — Web search for current events
_ = chat("What are the latest developments in AI agents in 2026?")


🧑 User: What are the latest developments in AI agents in 2026?

🤖 Agent Answer:


The latest developments in AI agents in 2026 involve the integration of various tools and frameworks to create more advanced and autonomous systems. Agentic AI, powered by LLMs, enables systems to plan and execute multi-step tasks, maintain state, backtrack on failure, and select the best strategy from multiple options. LangChain is a framework that enables building agents that can reason step-by-step using the ReAct paradigm, deciding which tools to call, executing actions, observing results, and iterating until a final answer is reached.

FAISS is a library for efficient similarity search and clustering of dense vectors, widely used as the backbone for vector databases in RAG systems. LlamaIndex is a data framework for building LLM-powered applications, providing connectors to ingest data from various sources, efficient indexing using vector stores, and query engines with sub-question decomposition, re-ranking, and hybrid search.

RAG is a technique that combines the strengths of large language models with external knowledge retrieval, fetching relevant documents from a vector store based on the user query and passing them as context to the LLM to generate a grounded, accurate response. This reduces hallucinations and keeps information up-to-date without expensive retraining.

I used the rag_knowledge_base tool to answer this question.

In [14]:
# Test 5 — Multi-turn conversation (memory test)
_ = chat("What is FAISS and why is it fast?")
_ = chat("How does it compare to Pinecone?")  # Should recall FAISS from previous turn


🧑 User: What is FAISS and why is it fast?

🤖 Agent Answer:


FAISS is a library for efficient similarity search and clustering of dense vectors. It supports billion-scale vector search and provides multiple index types: Flat (exact), IVF (approximate), and HNSW (graph-based). FAISS is highly optimized for GPU and CPU and is widely used as the backbone for vector databases in RAG systems.

FAISS is fast because it is highly optimized for GPU and CPU, allowing it to perform billion-scale vector search efficiently. It also provides multiple index types, which can be chosen based on the specific use case and requirements. The Flat index is exact, the IVF index is approximate, and the HNSW index is graph-based. This allows users to choose the trade-off between accuracy and speed.

In addition, FAISS is often used in conjunction with other tools and frameworks, such as RAG, LlamaIndex, and LangChain, to build more complex applications. These frameworks provide additional functionality, such as data ingestion, indexing, and query engines, which can be used in conjunction with FAISS to build more powerful and efficient systems.

Overall, FAISS is a powerful tool for efficient similarity search and clustering of dense vectors, and its speed advantages make it a popular choice for a wide range of applications.

Tools used: rag_knowledge_base


🧑 User: How does it compare to Pinecone?

🤖 Agent Answer:


RAG (Retrieval-Augmented Generation) and Pinecone are both used in the context of large language models (LLMs) and vector stores. However, they serve different purposes and have distinct functionalities.

RAG is a technique that combines the strengths of LLMs with external knowledge retrieval. It fetches relevant documents from a vector store based on the user query and passes them as context to the LLM to generate a grounded, accurate response. This reduces hallucinations and keeps information up-to-date without expensive retraining.

Pinecone, on the other hand, is a vector database that enables efficient similarity search and clustering of dense vectors. It supports billion-scale vector search and provides multiple index types, including Flat (exact), IVF (approximate), and HNSW (graph-based). Pinecone is highly optimized for GPU and CPU and is widely used as the backbone for vector databases in RAG systems.

In summary, RAG is a technique that leverages external knowledge retrieval to improve the accuracy and reliability of LLMs, while Pinecone is a vector database that enables efficient similarity search and clustering of dense vectors. While Pinecone can be used as a component in RAG systems, they are distinct concepts with different functionalities.

I used the rag_knowledge_base tool to answer this question.

## 🖥️ 8. (Optional) Gradio Chat UI

In [15]:
import gradio as gr
import uuid
from langchain_core.messages import HumanMessage, AIMessage

from langchain_core.messages import ToolMessage

def gradio_chat(message: str, history: list):
    try:
        result = agent.invoke(
            {"messages": [HumanMessage(content=message)]},
            config={"configurable": {"thread_id": str(uuid.uuid4())}}
        )

        # ✅ Only accept AIMessage with actual text content (not tool calls)
        answer = ""
        for msg in reversed(result["messages"]):
            if (
                isinstance(msg, AIMessage)          # must be AI, not Tool
                and isinstance(msg.content, str)    # must be plain string
                and msg.content.strip()             # must not be empty
                and not msg.tool_calls              # must not be a tool-calling step
            ):
                answer = msg.content
                break

        return answer or "⚠️ Agent finished but produced no text answer."

    except Exception as e:
        return f"❌ Error: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.ChatInterface(
        fn=gradio_chat,
        title="🤖 RAG Agentic Chatbot",
        description=(
            "Powered by **LlamaIndex** (FAISS vector store) + **LangChain ReAct Agent**\n"
            "Tools: 📚 RAG Knowledge Base | 🌐 Web Search | 📖 Wikipedia | 🔬 ArXiv"
        ),
        examples=[                                    # ✅ Gradio 6: string list only
            "What is RAG and how does LlamaIndex enable it?",
            "Find recent ArXiv papers on LLM agents",
            "Explain FAISS vector similarity search",
            "What is the ReAct prompting framework?",
        ],
        # ❌ type="messages" — removed in Gradio 6
        # ❌ theme=...       — moved to gr.Blocks()
    )

demo.launch(share=False, debug=True)

C:\Users\Lenovo-T460s\AppData\Local\Temp\ipykernel_5128\1269937743.py:31: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\Lenovo-T460s\Desktop\Rag_Agent\.venv\Lib\site-packages\gradio\routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\Lenovo-T460s\Desktop\Rag_Agent\.venv\Lib\site-packages\gradio\routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Keyboard interruption in main thread... closing server.


## 🗃️ 9. Add New Documents to the Index (Live Ingestion)

In [16]:
def add_documents(texts: list[str]) -> None:
    """
    Dynamically add new documents to the live FAISS index.
    No need to rebuild from scratch.
    """
    new_docs = [Document(text=t) for t in texts]
    for doc in new_docs:
        index.insert(doc)
    print(f"✅ Added {len(new_docs)} document(s). Total vectors: {faiss_index.ntotal}")

# Example: add a new document on-the-fly
add_documents([
    """Mixture of Experts (MoE) is an architecture where multiple specialized sub-networks 
    (experts) are trained, and a gating network selects which experts to activate per token. 
    This allows massive model capacity with sparse computation — only a fraction of parameters 
    are active per forward pass. GPT-4 and Mixtral both use MoE architectures."""
])

Applying transformations: 100%|██████████| 1/1 [00:00<00:00, 333.91it/s]

✅ Added 1 document(s). Total vectors: 6


## 📊 10. Architecture Summary

```
User Query
    │
    ▼
┌─────────────────────────────────────────┐
│         LangChain ReAct Agent           │
│  (Groq Llama-3 8B + ConversationBufferMemory)    │
└────────────────┬────────────────────────┘
                 │  selects tool(s)
    ┌────────────┼────────────────────┐
    │            │                   │
    ▼            ▼                   ▼
┌───────────┐ ┌─────────┐ ┌──────────────┐ ┌──────────┐
│  RAG Tool │ │   Web   │ │  Wikipedia   │ │  ArXiv   │
│           │ │ Search  │ │              │ │          │
│ LlamaIndex│ │DuckDuck │ │ LangChain    │ │LangChain │
│   FAISS   │ │   Go    │ │  Wrapper     │ │ Wrapper  │
│ VectorDB  │ │         │ │              │ │          │
└─────┬─────┘ └─────────┘ └──────────────┘ └──────────┘
      │
      ▼
  Top-K Similar
  Chunks (FAISS
  IndexFlatL2)
      │
      ▼
┌─────────────────────────────┐
│  Observations fed back to   │
│  Agent → Final Answer       │
└─────────────────────────────┘
```

### Key Design Decisions

| Component | Choice | Why |
|-----------|--------|-----|
| Vector Store | FAISS (IndexFlatL2) | Zero-latency exact search, no external service |
| Embeddings | HuggingFace `BAAI/bge-small-en-v1.5` | Free, runs locally, 384-dim |
| Agent Type | LangChain ReAct | Transparent reasoning, easy to debug |
| Memory | ConversationBufferWindow (k=10) | Balances context and token cost |
| Web Search | DuckDuckGo | No API key required |
| UI | Gradio ChatInterface | Quick deployment, shareable link |